In [7]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_park_by_grid(parks_shp, grid_shp, output_folder, resolution=10):
    # 读取数据
    parks = gpd.read_file(parks_shp)
    grid = gpd.read_file(grid_shp)

    # 坐标系统一（建议米制）
    if not parks.crs.is_projected:
        parks = parks.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(parks.crs)
    elif parks.crs != grid.crs:
        grid = grid.to_crs(parks.crs)

    # 清理空 geometry
    parks = parks[~parks.geometry.is_empty & parks.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_folder, exist_ok=True)

    for _, row in grid.iterrows():
        tile_id = row["ID"]  # 使用网格的 ID 字段作为文件名
        grid_geom = row.geometry

        # 提取该格网内的公园
        clipped = gpd.overlay(parks, gpd.GeoDataFrame(geometry=[grid_geom], crs=parks.crs), how="intersection")

        if clipped.empty:
            continue  # 如果该区域没有公园，跳过

        # 计算 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 准备 raster 数据
        shapes = [(geom, 1) for geom in clipped.geometry]
        raster = rasterize(
            shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype="uint8"
        )

        # 输出路径
        out_path = os.path.join(output_folder, f"park_tile_{tile_id}.tif")

        # 写入 raster
        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=parks.crs,
            transform=transform
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ Saved: {out_path}")

# 示例调用
rasterize_park_by_grid(
    parks_shp=r"D:\shanghai workshop\building+park+waterbody\landuse-公园用地\landuse_park.shp",
    grid_shp=rf"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    output_folder=r"D:\shanghai workshop\raster\final_park_tiles",
    resolution=2  # 2米分辨率
)


✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_196.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_532.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_539.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_882.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_883.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1225.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1226.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1568.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_1569.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3631.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3641.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3974.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\park_tile_3975.tif
✅ Saved: D:\shanghai workshop\raster\final_park_tiles\pa